# Medley PD — 팀원 시작 노트북 (Colab)

누구의 Drive 에도 의존하지 않습니다. **맨 위 셀의 `WORK` 만 각자 정하면** 코드·데이터·가중치를 모두 공개 저장소에서 받아 같은 결과를 재현합니다.

- 코드: GitHub (`REPO_URL`) · 데이터: OpenNeuro · CBraMod: GitHub + Hugging Face · S-CEReBrO: BioFoundation PR #18 (commit 361c017) + Hugging Face
- 고정 split 은 저장소의 `splits/` 를 팀 전체가 같이 씀 → 누가 돌려도 같은 환자 분할
- 피처는 같은 코드·같은 가중치면 똑같이 나옴 (검증: 기존 캐시와 최대 차이 0.0)

**런타임 → 런타임 유형 변경 → T4 GPU** 후 위에서부터 실행하세요. 피처 추출에만 GPU 가 필요합니다.

## 0. 각자 정할 것

In [ ]:
WORK = '/content/drive/MyDrive/medley_work'     # ✋ 내 작업 폴더 (피처·결과·모델 가중치). 코드 폴더와 분리!
DATA = '/content/data'                          # EEG 원본 (로컬 디스크가 빠름. 런타임 끝나면 사라지지만 피처는 WORK 에 남음)
REPO_URL = 'https://github.com/jeongwon8u8/Medley_skkai'
REPO_SUBDIR = 'medley_pd'                       # 저장소 안 코드 폴더

## 1. 준비 (Drive 연결, 코드 받기, 설치, 인코더 받기)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(WORK, exist_ok=True)
%cd /content
!rm -rf /content/repo
!git clone -q {REPO_URL} /content/repo
CODE = f'/content/repo/{REPO_SUBDIR}'
%cd {CODE}
!git -C /content/repo log -1 --format='코드 버전: %h %ci %s'
!pip install -q -r requirements.txt
!python run.py setup --work {WORK} --data {DATA}

In [ ]:
# (선택, 2~4분) 설치 확인: 가짜 데이터로 평가 파이프라인 전체 점검
!python tests/smoke_test.py 2>&1 | tail -1

## 2. 데이터 받기 (OpenNeuro)

In [ ]:
# PD-EEG: 안정 상태 기록만 (보행 기록 제외)
!python run.py download --dataset pd_eeg
# Iowa: PD 100명만
import pandas as pd
!python run.py download --dataset iowa_rest --set "dataset.download_include=[participants.tsv,participants.json,dataset_description.json,README]"
t = pd.read_csv(f'{DATA}/ds004584/participants.tsv', sep='\t')
inc = '[' + ','.join(['participants.tsv', 'participants.json'] + t.loc[t.GROUP.eq('PD'), 'participant_id'].tolist()) + ']'
!python run.py download --dataset iowa_rest --set "dataset.download_include={inc}" 2>&1 | tail -1
!du -sh {DATA}/*

## 3. 피처 추출 (GPU, 데이터셋 × 인코더)

In [ ]:
for ds in ['pd_eeg', 'iowa_rest']:
    for enc in ['cbramod', 'scerebro']:
        print(f'\n######## {ds} × {enc}')
        !python run.py extract --dataset {ds} --encoder {enc} 2>&1 | grep -v Warning | grep -v make_standard_montage | tail -4

## 4. 기관 내 평가 + 인코더 비교
`[split] 기존 고정 split 사용: …/splits/<dataset>__moca.csv` 가 나와야 팀 결과와 같은 분할입니다.

In [ ]:
for ds in ['pd_eeg', 'iowa_rest']:
    for enc in ['cbramod', 'scerebro']:
        !python run.py cv --dataset {ds} --encoder {enc} --task moca 2>&1 | grep -v Warning | sed -n '/대상/p;/split/p;/모델별 성능/,/paired/p'
    !python run.py compare --dataset {ds} --task moca --encoders cbramod scerebro

## 5. 기관 간 평가 (공통 변수 C0 = 나이·성별)

In [ ]:
for enc in ['cbramod', 'scerebro']:
    for tr, te in [('pd_eeg', 'iowa_rest'), ('iowa_rest', 'pd_eeg')]:
        !python run.py cross --train {tr} --test {te} --encoder {enc} --task moca 2>&1 | grep -v Warning | sed -n '/학습/p;/모델별 성능/,/paired/p'

## 6. 순열 검정 (결과가 우연 밖인지)
- `--mode label`: 정답을 섞음 → 모델 성능이 우연 밖인지 (기관 내 1회 약 2~4분 / 순열 100회)
- `--mode eeg`: EEG 만 섞음 → **EEG 의 추가 가치**(fusion − clinical)가 우연 밖인지
- 기관 간: 학습 기관의 정답을 섞어 다시 학습 (200회, 빠름)

비교 항목이 많아 5% 는 우연히 걸립니다. **주 검정은 미리 정한 것만** 결론에 씁니다: EEG-only Spearman (label), stack:C1 − clinical:C1 MAE (eeg), 기관 간 EEG-only Spearman (label).

In [ ]:
for ds in ['pd_eeg', 'iowa_rest']:
    for enc in ['cbramod', 'scerebro']:
        for mode in ['label', 'eeg']:
            !python run.py permtest --dataset {ds} --encoder {enc} --task moca --mode {mode} 2>&1 | grep -v Warning | sed -n '/== 순열/,$p'

In [ ]:
for enc in ['cbramod', 'scerebro']:
    for tr, te in [('pd_eeg', 'iowa_rest'), ('iowa_rest', 'pd_eeg')]:
        !python run.py permtest --dataset {tr} --test {te} --encoder {enc} --task moca 2>&1 | grep -v Warning | sed -n '/== 기관 간/,$p'

## 7. 요약 결과 모으기 → GitHub
`results_summary/` 에 요약 표(summary, paired, compare, permtest, 개인 경로를 지운 설정)만 모입니다. 환자별 예측·피처·원본은 포함하지 않습니다.

In [ ]:
!python run.py collect
!find results_summary -type f | head -40
!git -C /content/repo status --short | head -20

올리는 방법 (택 1)
- **웹**: `results_summary/` 와 `splits/` 를 내려받아 GitHub → Add file → Upload files
- **터미널(VS Code·서버)**: `git add results_summary splits && git commit -m "Add results" && git push`